In [1]:
!pip install -U langchain-ollama langchain-openai

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!ls /content/drive/MyDrive/review_dataset

sample_submission.csv  w_review_train.csv
test_file.csv	       w_review_train.parquet


In [4]:
import duckdb

con = duckdb.connect(database=':memory:', read_only=False)

In [5]:
df = con.execute("SELECT * FROM read_csv_auto('/content/drive/MyDrive/review_dataset/w_review_train.csv', parallel=false, encoding='UTF-8', ignore_errors=true)").fetchdf()
df.head()

,column0,column1
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,3
1,อาหารที่นี่เป็นอาหารจีนแคะที่หากินยากในบ้านเรา...,4
2,ปอเปี๊ยะสด ทุกวันนี้รู้สึกว่าหากินยาก (ร้านที่...,3
3,รัานคัพเค้กในเมืองไทยมีไม่มาก หลายๆคนอาจจะสงสั...,5
4,อร่อย!!! เดินผ่านDigital gatewayทุกวัน ไม่ยักร...,5


In [6]:
df.shape

(40000, 2)

In [7]:
con.execute("COPY df TO '/content/drive/MyDrive/review_dataset/w_review_train.parquet' (FORMAT PARQUET)")

In [8]:
parquet_avg_rating_direct = con.execute("SELECT AVG(column1) FROM '/content/drive/MyDrive/review_dataset/w_review_train.parquet' WHERE column0 LIKE '%อร่อย%'").fetchone()[0]
parquet_avg_rating_direct

3.8313027179006562

In [9]:
parquet_file_path = '/content/drive/MyDrive/review_dataset/w_review_train.parquet'

In [10]:
parquet_avg_rating_direct = con.execute(f"SELECT AVG(column1) FROM '{parquet_file_path}' WHERE column0 LIKE '%อร่อย%'").fetchone()[0]
parquet_avg_rating_direct

3.8313027179006562

In [11]:
coffee_keywords = [
    'กาแฟ', 'คาเฟ่', 'coffee', 'cafe', 'เอสเปรสโซ่', 'ลาเต้',
    'มอคค่า', 'คาปูชิโน่', 'เฟรนช์เพรส', 'cold brew', 'drip',
    'espresso', 'latte', 'americano', 'macchiato', 'brew',
    'specialty', 'slow bar', 'ร้านกาแฟ', 'ร้านคาเฟ่', 'กาแฟสด',
    'เมนูกาแฟ', 'กาแฟนม', 'กาแฟดำ', 'ร้านบาริสต้า', 'barista', 'coffee shop'
]

where_clause = " OR ".join([f"column0 LIKE '%{keyword}%'" for keyword in coffee_keywords])

In [12]:
%%time
coffee_reviews_df = con.execute(f"SELECT * FROM '{parquet_file_path}' WHERE {where_clause}").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 4.19 s, sys: 35 ms, total: 4.22 s
Wall time: 4.44 s


In [13]:
csv_file_path = '/content/drive/MyDrive/review_dataset/w_review_train.csv'

In [14]:
%%time
coffee_reviews_df_csv = con.execute(f"SELECT * FROM read_csv_auto('{csv_file_path}', parallel=false, encoding='UTF-8', ignore_errors=true) WHERE {where_clause}").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 4.31 s, sys: 46.1 ms, total: 4.35 s
Wall time: 2.4 s


In [15]:
coffee_reviews_df.head()

,column0,column1
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,3
1,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอา...,4
2,สารภาพว่าไม่เคยคิดจะไปต่อคิวซื้อมากินเองครับ บ...,3
3,ร้านอาหารญี่ปุ่นร้านนี้ ใจจริงไม่อยากแนะนำเลยค...,5
4,เดือนแรกที่เค้าต่อคิวกัน 2 - 3 ชั่วโมง เราก็ว่...,5


In [16]:
coffee_reviews_df.shape

(4981, 2)

In [17]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [18]:
!nohup ollama serve &

nohup: appending output to 'nohup.out'


In [19]:
!ollama pull gemma3:270m

Error: ollama server not responding - could not connect to ollama server, run 'ollama serve' to start it


In [20]:
!ollama list

NAME                                  ID              SIZE      MODIFIED       
scb10x/typhoon2.1-gemma3-4b:latest    8cfab9097c9d    2.6 GB    10 minutes ago    
gemma3:270m                           e7d36fb2c3b3    291 MB    12 minutes ago    


In [21]:
!ollama run gemma3:270m "สวัสดี"

สวัสดีค่ะ ยินดีที่ได้รู้จักค่ะ 😊




In [22]:
!curl -s http://127.0.0.1:11434/api/tags

{"models":[{"name":"scb10x/typhoon2.1-gemma3-4b:latest","model":"scb10x/typhoon2.1-gemma3-4b:latest","modified_at":"2025-11-11T05:19:41.783093305Z","size":2615662352,"digest":"8cfab9097c9d142fa16c7556826c9a7874f4642ac6d575e2d66a4f4979c48b76","details":{"parent_model":"","format":"gguf","family":"gemma3","families":["gemma3"],"parameter_size":"3.9B","quantization_level":"Q4_K_M"}},{"name":"gemma3:270m","model":"gemma3:270m","modified_at":"2025-11-11T05:17:26.615615714Z","size":291554930,"digest":"e7d36fb2c3b3293cfe56d55889867a064b3a2b22e98335f2e6e8a387e081d6be","details":{"parent_model":"","format":"gguf","family":"gemma3","families":["gemma3"],"parameter_size":"268.10M","quantization_level":"Q8_0"}}]}

In [23]:
from langchain_ollama import OllamaLLM
from  langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [24]:
llm = OllamaLLM(model="gemma3:270m")

In [25]:
response = llm.invoke("สวัสดี")
print(response)

สวัสดีครับหรือยังครับ? 😊



In [26]:
template = """
คุณเป็นครูสอน Python
คำถาม : {question}
"""

In [27]:
prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)

In [28]:
formatted_prompt = prompt.format(question="เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10")
print(formatted_prompt)


คุณเป็นครูสอน Python
คำถาม : เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10



In [29]:
chain = prompt | llm

In [30]:
response = chain.invoke({"question": "เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10"})

In [31]:
response

'```python\ndef find_sum_of_numbers(numbers):\n    """\n    หาผลรวมของตัวเลข 1 ถึง 10\n\n    Args:\n        numbers: จำนวนตัวเลขที่ต้องการหาผลรวมของ\n    Returns:\n        ผลรวมของตัวเลข 1 ถึง 10\n    """\n    try:\n        total = 0\n        for number in numbers:\n            total += number\n        return total\n    except Exception as e:\n        print(f"ข้อผิดพลาด: {e}")\n        return 0  # หรือจัดการกับค่าผิดพลาดในโค้ด\n    else:\n        return 0  # หรือจัดการกับค่าผิดพลาดในโค้ด\n```\n\n**คำอธิบายโค้ด:**\n\n1.  **`def find_sum_of_numbers(numbers):`**:  กำหนดชื่อ function ที่จะเรียกใช้ ซึ่งจะถูกเรียกโดยใช้เมื่อมีการเรียกใช้งานโค้ด\n2.  **`try...except`**:  เป็นการจัดการกับข้อผิดพลาดที่อาจเกิดขึ้น (เช่น การเรียกใช้งานโปรแกรมที่ไม่ถูกต้อง) โดยใช้ `try...except` เพื่อป้องกันการสูญเสียข้อมูล\n3.  **`total = 0`**:  เริ่มต้นค่า `total` เป็น 0 เพื่อเก็บผลรวมของตัวเลขที่ได้\n4.  **`for number in numbers:`**:  วนซ้ำผ่านทุกตัวเลขใน `numbers`  (ตัวเลข 1 ถึง 10)\n5.  **`total += number`**:

In [32]:
from IPython.display import display, Markdown

In [33]:
display(Markdown(response))

```python
def find_sum_of_numbers(numbers):
    """
    หาผลรวมของตัวเลข 1 ถึง 10

    Args:
        numbers: จำนวนตัวเลขที่ต้องการหาผลรวมของ
    Returns:
        ผลรวมของตัวเลข 1 ถึง 10
    """
    try:
        total = 0
        for number in numbers:
            total += number
        return total
    except Exception as e:
        print(f"ข้อผิดพลาด: {e}")
        return 0  # หรือจัดการกับค่าผิดพลาดในโค้ด
    else:
        return 0  # หรือจัดการกับค่าผิดพลาดในโค้ด
```

**คำอธิบายโค้ด:**

1.  **`def find_sum_of_numbers(numbers):`**:  กำหนดชื่อ function ที่จะเรียกใช้ ซึ่งจะถูกเรียกโดยใช้เมื่อมีการเรียกใช้งานโค้ด
2.  **`try...except`**:  เป็นการจัดการกับข้อผิดพลาดที่อาจเกิดขึ้น (เช่น การเรียกใช้งานโปรแกรมที่ไม่ถูกต้อง) โดยใช้ `try...except` เพื่อป้องกันการสูญเสียข้อมูล
3.  **`total = 0`**:  เริ่มต้นค่า `total` เป็น 0 เพื่อเก็บผลรวมของตัวเลขที่ได้
4.  **`for number in numbers:`**:  วนซ้ำผ่านทุกตัวเลขใน `numbers`  (ตัวเลข 1 ถึง 10)
5.  **`total += number`**:  เพิ่มค่า `number` เข้าไปใน `total`  (ค่าเพิ่มเข้าไปใน `total`  คือผลรวมของตัวเลขที่ได้)
6.  **`return total`**:  คืนค่า `total`  ซึ่งเป็นผลรวมของตัวเลขที่ได้
7.  **`except Exception as e:`**:  หากมีการเกิดข้อผิดพลาด (เช่น การเรียกใช้งานโปรแกรมที่ไม่ถูกต้อง)  จะใช้ `except Exception as e:`  เพื่อจัดการกับข้อผิดพลาดนี้  `e`  จะถูกนำมาใช้ในการตรวจสอบและจัดการข้อผิดพลาด
8.  **`else:`**:  ถ้าไม่มีข้อผิดพลาด (ไม่มีการเรียกใช้งานโปรแกรมที่ไม่ถูกต้อง) จะทำการคืนค่า `0`  (ค่าที่เก็บไว้ใน `total`  คือค่าที่ได้จากการหาผลรวม)
9.  **`return 0`**:  คืนค่าค่า `0`  (ค่าที่เก็บไว้ใน `total`  คือค่าที่ได้จากการหาผลรวม)

**ตัวอย่างการใช้งาน:**

```python
numbers = [1, 2, 3, 4, 5]
result = find_sum_of_numbers(numbers)
print(result)  # Output: 15
```

**ข้อดีของโค้ดนี้:**

*   **การจัดการข้อผิดพลาด:**  โค้ดนี้ใช้ `try...except` เพื่อจัดการกับข้อผิดพลาดที่อาจเกิดขึ้นจากการเรียกใช้งานโปรแกรมที่ไม่ถูกต้อง
*   **การจัดการค่าที่ได้:**  โค้ดนี้สามารถจัดการค่าที่ได้จาก `find_sum_of_numbers` ได้โดยไม่ต้องมีการเขียนโค้ดเอง
*   **การตรวจสอบข้อผิดพลาด:**  โค้ดนี้สามารถตรวจสอบข้อผิดพลาดที่อาจเกิดขึ้นในโค้ดอื่นๆ ได้

**ข้อควรระวัง:**

*   โค้ดนี้จะทำงานได้หากคุณใช้โปรแกรมที่ถูกต้องและมีข้อบกพร่องในการเรียกใช้งานโปรแกรมนั้น
*   การจัดการค่าที่ได้จาก `find_sum_of_numbers`  อาจต้องมีการปรับแต่งเพื่อให้เหมาะสมกับลักษณะของข้อมูลที่ใช้
*   การตรวจสอบข้อผิดพลาดในโค้ดนี้อาจเป็นเรื่องยาก



In [34]:
prompt1 = PromptTemplate(
    template="แปลโจทย์เกี่ยวกับ {question}"
)

chain1 = prompt1 | llm

In [35]:
prompt2 = PromptTemplate(
    template="ทำความเข้าใจโจทย์ที่แปลแล้วต่อไปนี้ {translate}"
)

chain2 = prompt2 | llm

In [36]:
prompt3 = PromptTemplate(
    template="แก้โจทย์ตามที่เข้าใจต่อไปนี้ {understand}"
)

chain3 = prompt3 | llm

In [37]:
prompt4 = PromptTemplate(
    template="อธิบายวิธิีทำจากผลลัทธ์ต่อไปนี้ {solve}"
)

chain4 = prompt4 | llm

In [38]:
parser_chain = StrOutputParser()

In [39]:
full_chain = chain1 | chain2 | chain3 | chain4 | parser_chain

In [40]:
result = full_chain.invoke({"question":"2x + 5 = 15"})

In [41]:
result

'ถูกต้องค่ะ โจทย์ "2x + 5 = 15" แปลเป็นคำตอบ **2x = 10**'

In [42]:
import os
from langchain_openai import ChatOpenAI

In [43]:
OPENAI_API_KEY = "xxx"
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [44]:
llm2 = ChatOpenAI(model="gpt-4o-mini")

In [45]:
chain1 = prompt1 | llm2
chain2 = prompt2 | llm2
chain3 = prompt3 | llm2
chain4 = prompt4 | llm2

full_chain = chain1 | chain2 | chain3 | chain4 | parser_chain

In [46]:
result = full_chain.invoke({"question":"2x + 5 = 15"})
result

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: xxx. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [47]:
formatted_result = result.replace('\\[', '$$') \
                         .replace('\\]', '$$') \
                         .replace('\\(', '$') \
                         .replace('\\)', '$')

In [48]:
display(Markdown(formatted_result))

ถูกต้องค่ะ โจทย์ "2x + 5 = 15" แปลเป็นคำตอบ **2x = 10**

In [49]:
prompt = """
Extract keywords from this review
{review}
"""

# Pros Simple, fast
# Cons อาจได้ผลไม่ดีกับ Tasks ซับซ้อน

In [50]:
prompt = """
Extract keywords from reviews.

Example 1
Review: "กาแฟอร่อย บริการดี"
Keywords: กาแฟ, บริการ

Example 2
Review: "ราคาแพง แต่คุ้มค่า"
Keywords: ราคา, คุณภาพ

Now extract from this review
Review: {review}
Keywords:
"""

# Pros ได้ผลดีกว่า Zero-shot
# Cons Prompt ใช้ Tokens มากขึ้น

In [51]:
prompt = """
Review: {review}

ขั้นตอนการ Extract keywords
1. อ่านรีวิวและเข้าใจความหมาย
2. ระบุหัวข้อหลักที่กล่าวถึง
3. เลือก 3 keywords ที่สำคัญที่สุด
4. ตอบในรูปแบบ JSON: {{"keywords": "คำ1, คำ2, คำ3"}}

ให้ทำทีละขั้นตอน
"""

# Pros ได้ผลดีกับ Reasoning tasks
# Cons ใช้ Tokens มากกว่า ช้ากว่า

In [52]:
keyword_prompt = PromptTemplate(
template = """
จาก Review ต่อไปนี้
Extract 3 keywords ที่เป็นตัวแทนของ Review นี้

Review: {review}

ขั้นตอนการ Extract keywords
1. อ่านรีวิวและเข้าใจความหมาย
2. ระบุหัวข้อหลักที่กล่าวถึง
3. เลือก 3 keywords ที่สำคัญที่สุด
4. ตอบในรูปแบบ JSON: {{"keywords": "คำ1, คำ2, คำ3"}}

ให้ทำทีละขั้นตอน
""")

In [53]:
sample_reviews = coffee_reviews_df['column0'].head(2)
sample_reviews


,column0
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...
1,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอา...


In [54]:
sample_reviews.to_csv('/content/sample_reviews.csv', index=False)

In [55]:
keyword_extract = keyword_prompt | llm | StrOutputParser()

In [56]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    cleaned = (text).replace("**", "").strip()
    results.append({"review": review, "text": cleaned})

CPU times: user 192 ms, sys: 24.7 ms, total: 217 ms
Wall time: 11.8 s


In [57]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

Review ร้านอาหารใหญ่มากกกกกกก 
เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี 
ห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ...
text ```json
{"keywords": ["ร้านอาหารใหญ่", "ห้องน้ำ", "กาแฟ", "น้ำผึ้ง", "น้ำผึ้ง", "ลาบไข่ต้ม", "ไข่มัน", "ขนมไทย", "ขนมหวาน", "ฟรี", "ราคา", "ซื้อ", "ไม่ไป", "แพง", "เกิน", "รับไม่ไหว", "กินแล้วมีขนมหวานให้กินฟรี", "ขนมไทย", "ขนมหวาน", "ไทย", "ร้านอาหาร", "อาหาร", "อาหารใหญ่", "ร้านอาหารใหญ่มาก", "เลี้ยว", "ห้องน้ำ", "ทานหลักๆ", "ชั้น 2", "กาแฟ", "น้ำผึ้ง", "น้ำผึ้งมาราด", "แพงเวอร์", "ราคาสูง", "ไม่แนะนำ", "ไม่สั่ง", "ลาบไข่ต้ม", "ไข่มันคาว", "คาว", "ไม่ประทับใจ", "ทอดมันหัวปลีกรอบอร่อย", "เบิ้ล", "มีขนมหวานให้กินฟรี", "ขนมไทย", "ขนมหวาน", "ไทย", "ร้านอาหาร", "อาหารใหญ่", "ร้านอาหารใหญ่มาก", "ร้านอาหารใหญ่มากมาก", "เลิกกินแล้ว", "ขนมหวานให้กินฟรี", "ขนมไทย", "ขนมหวาน", "ไทย", "ร้านอาหาร", "อาหาร", "อาหารใหญ่", "ร้านอาหารใหญ่มาก", "ร้านอาหารใหญ่มากมาก", "ร้านอาหารใหญ่มาก", "ร้านอาหารใหญ่มากมาก", "ร้านอาหารใหญ่มากมาก", "ร้านอาหารใหญ่มากมาก", "ร้านอาหารใหญ่มากมาก", "ร้านอาหาร

In [58]:
keyword_extract = keyword_prompt | llm2 | StrOutputParser()

In [59]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    cleaned = (text).replace("**", "").strip()
    results.append({"review": review, "text": cleaned})

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: xxx. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [60]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

In [61]:
!ollama pull scb10x/typhoon2.1-gemma3-4b

In [62]:
!ollama list

NAME                                  ID              SIZE      MODIFIED               
scb10x/typhoon2.1-gemma3-4b:latest    8cfab9097c9d    2.6 GB    Less than a second ago    
gemma3:270m                           e7d36fb2c3b3    291 MB    13 minutes ago            


In [63]:
llm3 = OllamaLLM(model="scb10x/typhoon2.1-gemma3-4b:latest")

In [64]:
keyword_extract = keyword_prompt | llm3 | StrOutputParser()

In [65]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    cleaned = (text).replace("**", "").strip()
    results.append({"review": review, "text": cleaned})

CPU times: user 121 ms, sys: 7.72 ms, total: 128 ms
Wall time: 17.7 s


In [66]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

Review ร้านอาหารใหญ่มากกกกกกก 
เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี 
ห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ...
text แน่นอนครับ ผมจะช่วยคุณ Extract keywords จากรีวิวที่ให้มา โดยจะทำตามขั้นตอนที่คุณระบุไว้

ขั้นตอนที่ 1: อ่านรีวิวและเข้าใจความหมาย

รีวิวนี้พูดถึงร้านอาหารขนาดใหญ่ที่มีจุดเริ่มต้นเป็นห้องน้ำ การเสิร์ฟกาแฟและน้ำผึ้งที่ไม่ได้มีอะไรพิเศษ ราคาอาหารค่อนข้างแพง โดยเฉพาะพะแนงห่อไข่ที่ราคา 150 บาท และมีขนมหวานให้ฟรีในตอนท้าย

ขั้นตอนที่ 2: ระบุหัวข้อหลักที่กล่าวถึง

*   ขนาดร้านอาหาร
*   ราคาอาหาร
*   รสชาติอาหาร (ลาบไข่ต้ม, ทอดมันหัวปลี, พะแนงห่อไข่)
*   ขนมหวานฟรี

ขั้นตอนที่ 3: เลือก 3 keywords ที่สำคัญที่สุด

จากหัวข้อหลักที่กล่าวถึง ผมเลือก keywords เหล่านี้:

1.  ราคาแพง
2.  รสชาติอาหาร
3.  ขนาดร้าน

ขั้นตอนที่ 4: ตอบในรูปแบบ JSON

```json
{
  "keywords": ["ราคาแพง", "รสชาติอาหาร", "ขนาดร้าน"]
}
```

หวังว่าคำตอบนี้จะเป็นประโยชน์นะครับ หากคุณมีรีวิวอื่นๆ ที่ต้องการให้ผมช่วย Extract keywords สามารถส่งมาได้เลย

Review วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์

ก็ได้ยิ

In [67]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

class ReviewKeywords(BaseModel):
    keywords: str = Field(
        ...,  # Required field
        description="3 keywords separated by comma",
        examples=["กาแฟ, บรรยากาศ, ราคา"]
    )

class KeywordsExtraction(BaseModel):
    extracted_qualities: ReviewKeywords

In [68]:
keywords_parser = JsonOutputParser(pydantic_object=KeywordsExtraction)

In [69]:
keyword_extract = keyword_prompt | llm3 | keywords_parser

In [70]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    results.append({"review": review, "text": text})

CPU times: user 139 ms, sys: 10.6 ms, total: 150 ms
Wall time: 12.3 s


In [71]:
results[0]['text']['keywords']

['ราคา', 'อาหาร', 'ขนาด']

In [72]:
results[1]['text']['keywords']

['กาแฟ', 'บรรยากาศ', 'รสชาติ']

In [73]:
llm3 = OllamaLLM(
    model="scb10x/typhoon2.1-gemma3-4b",
    temperature=0.05
)

keyword_extract = keyword_prompt | llm3 | keywords_parser

In [74]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    results.append({"review": review, "text": text})

CPU times: user 151 ms, sys: 11.5 ms, total: 162 ms
Wall time: 12.9 s


In [75]:
results[0]['text']['keywords'], results[1]['text']['keywords']

(['ราคา', 'อาหาร', 'ขนาด'], ['กาแฟ', 'บรรยากาศ', 'รสชาติ'])

In [76]:
keyword_prompt = PromptTemplate(
    template="""จาก Review ต่อไปนี้
    Extract keywords 3 keywords เท่านั้น ที่เป็นตัวแทนของ Review นี้ได้ดีที่สุด

    Review: {review}

    ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'keywords'
    และค่าเป็นข้อความที่ดึงได้คั่นด้วย comma
    เช่น {{"keywords": "กาแฟ, บรรยากาศ, ราคา"}}

    หากไม่พบข้อมูลให้ตอบ 'ไม่ระบุ'
    """
)

In [77]:
keyword_extract = keyword_prompt | llm3 | keywords_parser

In [78]:
inputs = [{"review": r} for r in sample_reviews]

In [79]:
%%time
results = keyword_extract.batch(inputs)

CPU times: user 19.7 ms, sys: 3.96 ms, total: 23.7 ms
Wall time: 2.47 s


In [80]:
results

[{'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
 {'keywords': 'กาแฟ, บรรยากาศ, รสชาติ'}]

In [81]:
sample_reviews_1000 = coffee_reviews_df['column0'].head(1000).tolist()
inputs_1000 = [{"review": r} for r in sample_reviews_1000]

BATCH_SIZE = 8

results_1000 = []

In [82]:
from tqdm import tqdm

In [83]:
for i in tqdm(range(0, len(inputs_1000), BATCH_SIZE), desc="Extracting keywords"):
    batch = inputs_1000[i:i+BATCH_SIZE]
    outputs = keyword_extract.batch(batch)
    for inp, out in zip(batch, outputs):
        results_1000.append({"review": inp["review"], "text": out})

Extracting keywords: 100%|██████████| 125/125 [20:38<00:00,  9.91s/it]


In [84]:
for result in results_1000[0:10]:
    print(result['text']['keywords'])

ราคา, บรรยากาศ, ไข่ต้ม
กาแฟ, บรรยากาศ, รสชาติ
โดนัท, รสชาติ, ความหวาน
วาซาบิ, ปลาซาบะ, ข้าวหน้าปลา
โดนัท, Original Glazed, คิว
วังพญาไท, นรสิงห์, คาเฟ่
อาหารทะเล, ราคา, บรรยากาศ
กาแฟ, เค้ก, พนักงาน
ฮันนี่ โทสท์, ราคา, รสชาติ
คาเฟ่, วานิลลา, คาโบนาร่า


In [85]:
context_prompt = PromptTemplate(
    template="""จาก Review ต่อไปนี้
    Extract context ใน Review ที่เป็นตัวแทนที่บรรยายคำว่า {keyword} ได้ดีที่สุด

    Review: {review}

    ตอบผลลัพธ์ในรูปแบบ JSON โดยมีคีย์เป็น 'context'
    และค่าเป็นข้อความที่ดึงได้จาก Review

    ตัวอย่าง
    keyword คือ "กาแฟ"
    context: "กาแฟที่นี่อร่อยมาก หอมกลิ่นกาแฟคั่วสด"

    หากไม่พบข้อมูลให้ตอบ 'ไม่ระบุ'
    """
)

In [86]:
sentiment_prompt = PromptTemplate(
    template="""จากข้อความต่อไปนี้ ให้วิเคราะห์ว่าเป็น sentiment แบบไหน
    เลือกจากตัวเลือกเหล่านี้เท่านั้น: "positive", "negative", "neutral"

    ข้อความ: {message}

    ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'sentiment'
    และค่าเป็นหนึ่งใน list ต่อไปนี้ [positive, negative, neutral]

    คำแนะนำ
    - positive คือ ข้อความที่แสดงความพอใจ ชอบ ดี
    - negative คือ ข้อความที่แสดงความไม่พอใจ ไม่ชอบ แย่
    - neutral คือ ข้อความที่ไม่แสดงความรู้สึกชัดเจน เป็นกลาง

    # ตัวอย่าง
    # ข้อความ "อร่อยมาก" → {{"sentiment": "positive"}}
    # ข้อความ "แพงไป" → {{"sentiment": "negative"}}
    # ข้อความ "ปกติ" → {{"sentiment": "neutral"}}
    """
)

In [87]:
class ContextKeyword(BaseModel):
    context: str = Field(..., description="context that best represent this keyword")

class ContexExtraction(BaseModel):
    extracted: ContextKeyword

context_parser = JsonOutputParser(pydantic_object=ContexExtraction)

In [88]:
context_extract = context_prompt | llm3 | context_parser

In [89]:
class Sentiment(BaseModel):
    sentiment: str = Field(..., description="sentiment of review")

class SentimentExtraction(BaseModel):
    extracted: Sentiment


sentiment_parser = JsonOutputParser(pydantic_object=SentimentExtraction)

In [90]:
sentiment_extract = sentiment_prompt | llm3 | sentiment_parser

In [91]:
results_1000[0]['text']['keywords'].split(',')

['ราคา', ' บรรยากาศ', ' ไข่ต้ม']

In [92]:
context_inputs = []

for review_data in tqdm(results_1000, desc="Preparing context inputs"):
    review = review_data['review']
    keywords = review_data['text']['keywords'].split(',')
    for keyword in keywords:
        context_inputs.append({"review": review, "keyword": keyword.strip()})

Preparing context inputs: 100%|██████████| 1000/1000 [00:00<00:00, 210040.76it/s]


In [93]:
len(context_inputs)

3000

In [94]:
context_results = []
for i in tqdm(range(len(results_1000)), desc="Extracting contexts"):
   result = context_extract.batch(context_inputs[i*3:(i*3)+3])
   contexts_list = [d['context'] for d in result]
   results_1000[i]['contexts'] = contexts_list

Extracting contexts: 100%|██████████| 1000/1000 [1:17:05<00:00,  4.63s/it]


In [95]:
results_1000[:2]

[{'review': 'ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี \nห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย \nลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่\nทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล \nพะแนงห่อไข่อร่อยดี เห้ยแต่ราคา 150บาทมันเกินไปนะ รับไม่ไหวว\nเลิกกินแล้วมีขนมหวานให้กินฟรีเล็กน้อย )ขนมไทย) \n\nคงไม่ไปซ้ำ แพงเกิน ',
  'text': {'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
  'contexts': ['ราคา 150 บาทมันเกินไปนะ รับไม่ไหวว',
   'บรรยากาศร้านใหญ่มากกกกกกก',
   'ลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่']},
 {'review': 'วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์\n\nก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก Twitter ว่ากาแฟที่นี่อร่อยมากกกกกกก\nเรียกว่าระดับแฟนๆ Starbuck ยังต้องเหลียวมามอง ก็ดองมานานจนถึงวันนี้โอกาสเหมาะ ไป\npresent งานที่ตึก Software Park เสร็จก็เลยมาหาไรกินแถวนี้พอดี\nจัดไป.....\n\n\nบรรยากาศรอบๆร้านก็แต่งได้น่านั่ง อารมณ์ประมาณว่าอยู่ในสวนบ้านเราเอง ต้นไม้เยอะดี\nอย่างน้อยก็เป็นสีเขียวเล็กๆ ในใจกลางเม

In [96]:
sentiment_inputs = []

for review_data in tqdm(results_1000, desc="Preparing sentiment inputs"):
    contexts = review_data['contexts']
    for context in contexts:
        sentiment_inputs.append({"message": context})

Preparing sentiment inputs: 100%|██████████| 1000/1000 [00:00<00:00, 286261.53it/s]


In [97]:
len(sentiment_inputs)

3000

In [98]:
sentiment_inputs[:6]

[{'message': 'ราคา 150 บาทมันเกินไปนะ รับไม่ไหวว'},
 {'message': 'บรรยากาศร้านใหญ่มากกกกกกก'},
 {'message': 'ลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่'},
 {'message': 'กาแฟที่นี่อร่อยมากกกกกกก เรียกว่าระดับแฟนๆ Starbuck ยังต้องเหลียวมามอง'},
 {'message': 'บรรยากาศรอบๆร้านก็แต่งได้น่านั่ง อารมณ์ประมาณว่าอยู่ในสวนบ้านเราเอง ต้นไม้เยอะดี อย่างน้อยก็เป็นสีเขียวเล็กๆ ในใจกลางเมืองได้ละ เขียวชอุ่ม'},
 {'message': 'รสชาติและกลิ่นมันไม่ธรรมดาเลย หอมๆ ถือว่าเป็นมวยรุ่นเดียวกะ Starbuck ได้'}]

In [102]:
for i in tqdm(range(len(results_1000)), desc="Extracting sentiment"):
   result = sentiment_extract.batch(sentiment_inputs[i*3:(i*3)+3])
   sentiment_list = [d['sentiment'] for d in result]
   results_1000[i]['sentiment'] = sentiment_list

Extracting sentiment: 100%|██████████| 1000/1000 [44:22<00:00,  2.66s/it]


In [103]:
results_1000[0:2]

[{'review': 'ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี \nห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย \nลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่\nทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล \nพะแนงห่อไข่อร่อยดี เห้ยแต่ราคา 150บาทมันเกินไปนะ รับไม่ไหวว\nเลิกกินแล้วมีขนมหวานให้กินฟรีเล็กน้อย )ขนมไทย) \n\nคงไม่ไปซ้ำ แพงเกิน ',
  'text': {'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
  'contexts': ['ราคา 150 บาทมันเกินไปนะ รับไม่ไหวว',
   'บรรยากาศร้านใหญ่มากกกกกกก',
   'ลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่'],
  'sentiment': ['negative', 'positive', 'negative']},
 {'review': 'วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์\n\nก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก Twitter ว่ากาแฟที่นี่อร่อยมากกกกกกก\nเรียกว่าระดับแฟนๆ Starbuck ยังต้องเหลียวมามอง ก็ดองมานานจนถึงวันนี้โอกาสเหมาะ ไป\npresent งานที่ตึก Software Park เสร็จก็เลยมาหาไรกินแถวนี้พอดี\nจัดไป.....\n\n\nบรรยากาศรอบๆร้านก็แต่งได้น่านั่ง อารมณ์ประมาณว่าอยู่ในสวนบ้านเราเอง

In [106]:
import pandas as pd

df = pd.DataFrame([
    {
        'review': r['review'],
        **{f'keyword{i+1}': k.strip() for i, k in enumerate(r['text']['keywords'])},
        **{f'context{i+1}_keyword': c for i, c in enumerate(r['contexts'])},
        **{f'sentiment{i+1}': s for i, s in enumerate(r['sentiment'])}
    }
    for r in results_1000
])

In [107]:
df.to_csv('results.csv', index=False, encoding='utf-8-sig')